# 4.4 📊 Ingestión Analítica: De API → JSON → DataFrame

Hasta ahora aprendimos a:

- Consumir APIs
- Diseñar Providers desacoplados
- Gestionar autenticación, retries y cache

Pero aún estamos en formato **operacional (JSON crudo)**.

Para análisis, modelado y BI necesitamos transformar esos datos a estructuras analíticas.

> Esta sección conecta el mundo API con el mundo Data Science.


## 🧭 Pipeline de Ingestión Analítica

```{mermaid}
flowchart LR
    A[External API] --> B[JSON Response]
    B --> C[Parsing Layer]
    C --> D[Normalization]
    D --> E[DataFrame]
    E --> F[Feature Engineering / BI]
```


## 🧠 Problema: JSON no es tabular

Las APIs devuelven estructuras anidadas que debemos transformar a tablas analíticas.


In [ ]:
import pandas as pd

api_response = {
    "symbol": "AAPL",
    "prices": [
        {"date": "2024-01-01", "close": 150},
        {"date": "2024-01-02", "close": 152},
        {"date": "2024-01-03", "close": 149},
    ],
}

api_response

## 🔄 Normalización JSON → DataFrame

In [ ]:
df = pd.json_normalize(api_response["prices"])
df

## 🕒 Manejo de fechas

In [ ]:
df["date"] = pd.to_datetime(df["date"])
df = df.set_index("date")
df

## 🧩 Manejo de valores faltantes

In [ ]:
df_missing = df.copy()
df_missing.loc["2024-01-02", "close"] = None
df_missing

In [ ]:
df_missing["close_ffill"] = df_missing["close"].ffill()
df_missing

## 🔗 Integración multi‑fuente

```{mermaid}
flowchart LR
    A[Prices API] --> D[Merge Layer]
    B[FX API] --> D
    C[Macro API] --> D
    D --> E[Unified DataFrame]
```


In [ ]:
prices = pd.DataFrame({
    "date": pd.date_range("2024-01-01", periods=3),
    "close": [150, 152, 149],
})

fx = pd.DataFrame({
    "date": pd.date_range("2024-01-01", periods=3),
    "usd_cop": [4000, 4020, 3980],
})

df_merged = prices.merge(fx, on="date")
df_merged

## ⚙️ Feature Engineering inicial

In [ ]:
df_merged["return"] = df_merged["close"].pct_change()
df_merged

## 🏗️ Arquitectura Analítica Completa

```{mermaid}
flowchart LR
    A[Providers] --> B[Ingestion Layer]
    B --> C[Normalization]
    C --> D[Data Lake / Warehouse]
    D --> E[Feature Store]
    E --> F[Models / BI]
```


## 🎯 Cierre

La ingestión analítica transforma datos operacionales en datos listos para análisis y modelado.
